In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import json
import os

Configuration

In [ ]:
config_path = Path("../config/config.json")
config = json.load(open(config_path))

Finding paths of files for analysis

In [ ]:
position_file_name = 'zmiana_pozycji.csv'
baseline_file_name = 'baseline.csv'

position_files_paths = []
baseline_files_paths = []

In [ ]:
data_folder_path = Path(config["data_folder_path"])
print(data_folder_path)

patient_folders = os.listdir(data_folder_path)

for pf in patient_folders:
    if "PAC" not in pf:
        continue

    patient_folder_path = data_folder_path / pf
    patient_files = os.listdir(patient_folder_path)

    position_file_path = patient_folder_path / position_file_name
    baseline_file_path = patient_folder_path / baseline_file_name

    position_files_paths.append(position_file_path)
    baseline_files_paths.append(baseline_file_path)

Check if all types of files are available for each patient

In [ ]:
print(f"Number of position files: {len(position_files_paths)}")
print(f"Number of baseline files: {len(baseline_files_paths)}")

Analysis of body position change experiment files

In [ ]:
fv_rs = []
fv_ls = []
abps_finger = []
abps_arm = []
rrs = []

In [ ]:
for pf in position_files_paths:
    df = pd.read_csv(pf, sep=";")
    fv_rs.extend(df["fv_r"].values)
    fv_ls.extend(df["fv_l"].values)
    abps_finger.extend(df["abp_finger[mm_Hg]"].values)
    abps_arm.extend(df["abp_arm[mm_Hg]"].values)
    rrs.extend(df["rr[bpm]"].values)

Change of data type from str to int

In [ ]:
fv_rs = [float(x.replace(',', '.')) if isinstance(x, str) else float(x) for x in fv_rs]
fv_ls = [float(x.replace(',', '.')) if isinstance(x, str) else float(x) for x in fv_ls]
abps_finger = [float(x.replace(',', '.')) if isinstance(x, str) else float(x) for x in abps_finger]
abps_arm = [float(x.replace(',', '.')) if isinstance(x, str) else float(x) for x in abps_arm]
rrs = [float(x.replace(',', '.')) if isinstance(x, str) else float(x) for x in rrs]

Analyzing signal statistics for all subjects

In [ ]:
def plot_column_distribution(column_name, df):
    plt.figure(figsize=(10, 6))
    plt.hist(df[column_name], bins=30, edgecolor="k", alpha=0.7)
    plt.title(f"Distribution of {column_name}")
    plt.xlabel(column_name)
    plt.ylabel("Frequency")
    plt.grid(axis="y", alpha=0.75)
    plt.show()

In [ ]:
def analyze_basic_features(column_name, df):
    print(f"Analyzing {column_name}...")
    print(f"Mean: {df[column_name].mean()}")
    print(f"Median: {df[column_name].median()}")
    print(f"Standard Deviation: {df[column_name].std()}")
    print(f"Min: {df[column_name].min()}")
    print(f"Max: {df[column_name].max()}")
    print(f"Number of missing values: {df[column_name].isna().sum()}")
    print("---------------------------------------------------------------")
    

In [ ]:
analyze_basic_features("fv_r", pd.DataFrame({"fv_r": fv_rs}))
analyze_basic_features("fv_l", pd.DataFrame({"fv_l": fv_ls}))
analyze_basic_features("abp_finger[mm_Hg]", pd.DataFrame({"abp_finger[mm_Hg]": abps_finger}))
analyze_basic_features("abp_arm[mm_Hg]", pd.DataFrame({"abp_arm[mm_Hg]": abps_arm}))
analyze_basic_features("rr[bpm]", pd.DataFrame({"rr[bpm]": rrs}))

In [ ]:
plot_column_distribution("fv_r", pd.DataFrame({"fv_r": fv_rs}))
plot_column_distribution("fv_l", pd.DataFrame({"fv_l": fv_ls}))
plot_column_distribution("abp_finger[mm_Hg]", pd.DataFrame({"abp_finger[mm_Hg]": abps_finger}))
plot_column_distribution("abp_arm[mm_Hg]", pd.DataFrame({"abp_arm[mm_Hg]": abps_arm}))
plot_column_distribution("rr[bpm]", pd.DataFrame({"rr[bpm]": rrs}))

Analysis for each subject

In [ ]:
column_names = ["fv_r", "fv_l", "abp_finger[mm_Hg]", "abp_arm[mm_Hg]", "rr[bpm]"]

In [ ]:
def plot_all_signals_for_patient(patient_df, patient_id):
    patient_df = patient_df.sort_values('timestamp')

    fig, axes = plt.subplots(len(column_names), 1, figsize=(15, 10), sharex=True)

    if len(column_names) == 1:
        axes = [axes]

    for i, column in enumerate(column_names):

        axes[i].plot(patient_df.index, patient_df[column])
        axes[i].set_ylabel(column)
        axes[i].set_title(f"{column} over time, patient {patient_id}")
        axes[i].grid(True)

    axes[-1].set_xlabel("Time")

    plt.tight_layout()
    plt.show()

In [ ]:
def get_patient_number_from_path(file_path):
    return file_path.parts[8]

# pth = position_files_paths[0]
# print(pth)
# patient_id = get_patient_number_from_path(pth)
# print(f"Patient ID: {patient_id}")

In [ ]:
for pf in position_files_paths:
    patient_id = get_patient_number_from_path(pf)
    df = pd.read_csv(pf, sep=";")
    df['timestamp'] = pd.to_datetime(
    df['DateTime'].str.replace(',', '.').astype(float),
    origin='1899-12-30',
    unit='D'
    )
    df = df.dropna(subset=['timestamp']).sort_values('timestamp')
    for col in column_names:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.'),
            errors='coerce'
        )

    df = df.set_index('timestamp').interpolate()

    df = df.sort_index()

    df[column_names] = df[column_names].rolling('5s', min_periods=1).mean()

    plot_all_signals_for_patient(df, patient_id)
    